In [ ]:
# Cell 1: Cài đặt các thư viện cần thiết trên môi trường Kaggle / Google Colab
!pip install -q torch-geometric mediapipe opencv-python tensorboard tqdm pyyaml

In [ ]:
# Cell 2: Import các thư viện cần thiết (sử dụng tqdm mặc định, không dùng tqdm của notebook)
from __future__ import annotations
import csv
import json
import logging
import math
import os
import random
import sys
import tempfile
import time
import urllib.request
from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import List, Tuple, Dict, Any, Optional

# Đảm bảo mã hóa UTF-8 trên console
if hasattr(sys.stdout, "reconfigure"):
    try:
        sys.stdout.reconfigure(encoding="utf-8", errors="replace")
    except Exception:
        pass
if hasattr(sys.stderr, "reconfigure"):
    try:
        sys.stderr.reconfigure(encoding="utf-8", errors="replace")
    except Exception:
        pass

import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter
from torch_geometric.nn import DenseGCNConv, DenseGATConv
from torch_geometric.utils import to_undirected, add_self_loops, to_dense_adj
from tqdm import tqdm

logger = logging.getLogger("DriverGuardian.GCN")

In [ ]:
# Cell 3: Hệ thống cấu hình trung tâm (config.py)
@dataclass
class TrainConfig:
    """
    Cấu hình tổng hợp toàn hệ thống Driver Guardian GCN.
    Tất cả các siêu tham số được quản lý tập trung và phân nhóm rõ ràng.
    """
    # =====================================================================
    # 1. CẤU HÌNH ĐỒ THỊ VÀ TRÍCH XUẤT ĐỘNG HỌC (GRAPH & KINEMATICS)
    # =====================================================================
    num_nodes: int = 68
    nose_idx: int = 30           # Mốc tâm chóp mũi (gốc tọa độ cục bộ)
    left_eye_idx: int = 36       # Khóe mắt ngoài trái (đo IOD)
    right_eye_idx: int = 45      # Khóe mắt ngoài phải (đo IOD)
    eps: float = 1e-6            # Tránh chia cho 0 khi tính khoảng cách IOD
    adaptive_init_scale: float = 1e-5  # Hệ số khởi tạo ma trận kề thích ứng

    # Cặp chỉ số đo độ mở mắt (EAR) & miệng (MAR)
    vertical_pairs: List[Tuple[int, int]] = field(default_factory=lambda: [
        (37, 41), (38, 40), (43, 47), (44, 46),  # Mí mắt trên - dưới
        (50, 58), (51, 57), (52, 56), (61, 67), (62, 66), (63, 65)  # Môi trên - dưới
    ])

    # Cặp chỉ số tiên nghiệm liên vùng (Mắt <-> Miệng, Mũi <-> Cằm)
    cross_region_pairs: List[Tuple[int, int]] = field(default_factory=lambda: [
        (37, 48), (41, 48), (38, 62), (40, 66),  # Mắt trái <-> Miệng
        (44, 54), (46, 54), (43, 62), (47, 66),  # Mắt phải <-> Miệng
        (30, 8), (30, 36), (30, 45), (27, 8)     # Chóp mũi <-> Cằm & Đuôi mắt
    ])

    # =====================================================================
    # 2. CẤU HÌNH MÔ HÌNH (MODEL ARCHITECTURE)
    # =====================================================================
    model_name: str = "stgcn"                 # "stgcn" | "transformer"
    in_channels: int = 6                      # [x, y, dx, dy, d2x, d2y]
    num_classes: int = 2                      # Mặc định 2 lớp (Alert vs Drowsy)

    # Siêu tham số riêng cho Model 1: DriverSTGCN
    stgcn_hidden_dims: List[int] = field(default_factory=lambda: [64, 128, 256])
    stgcn_strides: List[int] = field(default_factory=lambda: [1, 2, 2])
    temporal_kernel: int = 5                  # Độ dài kernel tích chập thời gian
    gat_heads: int = 4                        # Số attention heads của DenseGATConv
    dropout: float = 0.25                     # Tỷ lệ Dropout cho STGCN
    classifier_hidden_dim: Optional[int] = None  # None: mặc định curr_dim // 2

    # Siêu tham số riêng cho Model 2: DriverSTGCNTransformer
    d_model: int = 128                        # Kích thước biểu diễn của Transformer
    nhead: int = 4                            # Số Attention Heads trong Transformer
    num_transformer_layers: int = 2           # Số lớp TransformerEncoder
    max_len: int = 300                        # Chiều dài chuỗi thời gian tối đa
    transformer_dropout: float = 0.2
    stem_channels: List[int] = field(default_factory=lambda: [64, 128])
    stem_strides: List[int] = field(default_factory=lambda: [1, 2])
    dim_feedforward: Optional[int] = None     # None: mặc định d_model * 2
    init_std: float = 0.02                    # Độ lệch chuẩn khởi tạo CLS token & PosEmbedding

    # =====================================================================
    # 3. CẤU HÌNH BỘ DỮ LIỆU (DATASET CONFIG)
    # =====================================================================
    dataset_dir: str = r"E:\LSTM\data_processed"
    manifest_filename: str = "dataset_merged_split.csv"
    time_interval: float = 0.2                # 0.2s = 5 FPS lấy mẫu thời gian thực
    window_sec: Optional[float] = 10.0        # Cửa sổ thời gian (giây), None: toàn bộ video
    landmark_model_path: Optional[str] = None # None: tự tìm face_landmarker.task trong project
    class_names: Dict[int, str] = field(default_factory=lambda: {
        0: "Alert",
        1: "Drowsy"
    })
    letterbox_size: int = 480

    # =====================================================================
    # 4. CẤU HÌNH DATALOADER & HUẤN LUYỆN (DATALOADER & TRAINING)
    # =====================================================================
    batch_size: int = 16
    num_workers: int = 0
    pin_memory: bool = True
    shuffle_train: bool = True

    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    lr: float = 1e-3
    weight_decay: float = 1e-4
    epochs: int = 50
    warmup_epochs: int = 5
    min_lr_ratio: float = 0.05
    max_grad_norm: float = 1.0
    amp: bool = True
    save_dir: str = "ai/GCN/checkpoints"
    log_dir: Optional[str] = None
    resume: Optional[str] = None
    seed: int = 42

    def __post_init__(self):
        """Tự động điều chỉnh và đồng bộ các thuộc tính phụ thuộc."""
        if self.class_names is None or len(self.class_names) != self.num_classes:
            if self.num_classes == 2:
                self.class_names = {0: "Alert", 1: "Drowsy"}
            elif self.num_classes == 3:
                self.class_names = {0: "Alert", 1: "Fatigue", 2: "Drowsy"}
            else:
                self.class_names = {i: f"Class_{i}" for i in range(self.num_classes)}

    def to_dict(self) -> Dict[str, Any]:
        """Chuyển đổi toàn bộ cấu hình sang dictionary."""
        return asdict(self)

    def save(self, path: str | Path) -> None:
        """Lưu cấu hình sang tệp JSON hoặc YAML (nếu có PyYAML)."""
        p = Path(path)
        p.parent.mkdir(parents=True, exist_ok=True)
        data = self.to_dict()

        if p.suffix.lower() in (".yaml", ".yml"):
            try:
                import yaml
                with open(p, "w", encoding="utf-8") as f:
                    yaml.safe_dump(data, f, sort_keys=False, allow_unicode=True)
                return
            except ImportError:
                pass  # Fallback to json if PyYAML is not installed

        with open(p, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=4, ensure_ascii=False)

    @classmethod
    def load(cls, path: str | Path) -> "TrainConfig":
        """Nạp cấu hình từ tệp JSON hoặc YAML."""
        p = Path(path)
        if not p.is_file():
            raise FileNotFoundError(f"Không tìm thấy file config tại: {p}")

        content = None
        if p.suffix.lower() in (".yaml", ".yml"):
            try:
                import yaml
                with open(p, "r", encoding="utf-8") as f:
                    content = yaml.safe_load(f)
            except ImportError:
                pass

        if content is None:
            with open(p, "r", encoding="utf-8") as f:
                content = json.load(f)

        # Chuyển class_names key sang int nếu nạp từ json (json lưu key dạng string)
        if "class_names" in content and isinstance(content["class_names"], dict):
            content["class_names"] = {int(k): v for k, v in content["class_names"].items()}

        return cls(**content)

    # -----------------------------------------------------------------
    # Helper methods trích xuất kwargs để truyền vào các components
    # -----------------------------------------------------------------
    def get_stgcn_kwargs(self) -> Dict[str, Any]:
        """Trích xuất tham số thuần túy cho mô hình DriverSTGCN."""
        return {
            "in_channels": self.in_channels,
            "num_classes": self.num_classes,
            "num_nodes": self.num_nodes,
            "hidden_dims": list(self.stgcn_hidden_dims),
            "strides": list(self.stgcn_strides),
            "dropout": self.dropout,
            "temporal_kernel": self.temporal_kernel,
            "gat_heads": self.gat_heads,
            "classifier_hidden_dim": self.classifier_hidden_dim,
            "nose_idx": self.nose_idx,
            "left_eye_idx": self.left_eye_idx,
            "right_eye_idx": self.right_eye_idx,
            "eps": self.eps,
        }

    def get_transformer_kwargs(self) -> Dict[str, Any]:
        """Trích xuất tham số thuần túy cho mô hình DriverSTGCNTransformer."""
        return {
            "in_channels": self.in_channels,
            "num_classes": self.num_classes,
            "num_nodes": self.num_nodes,
            "d_model": self.d_model,
            "nhead": self.nhead,
            "num_transformer_layers": self.num_transformer_layers,
            "max_len": self.max_len,
            "dropout": self.transformer_dropout,
            "stem_channels": list(self.stem_channels),
            "stem_strides": list(self.stem_strides),
            "dim_feedforward": self.dim_feedforward,
            "init_std": self.init_std,
            "nose_idx": self.nose_idx,
            "left_eye_idx": self.left_eye_idx,
            "right_eye_idx": self.right_eye_idx,
            "eps": self.eps,
        }

    def get_dataset_kwargs(self, split: str = "train") -> Dict[str, Any]:
        """Trích xuất tham số thuần túy cho GCNDataset."""
        return {
            "dataset_dir": self.dataset_dir,
            "split": split,
            "time_interval": self.time_interval,
            "window_sec": self.window_sec,
            "manifest_filename": self.manifest_filename,
            "landmark_model_path": self.landmark_model_path,
            "letterbox_size": self.letterbox_size,
        }

    def get_dataloader_kwargs(self) -> Dict[str, Any]:
        """Trích xuất tham số thuần túy cho build_gcn_dataloaders."""
        return {
            "dataset_dir": self.dataset_dir,
            "batch_size": self.batch_size,
            "num_workers": self.num_workers,
            "time_interval": self.time_interval,
            "window_sec": self.window_sec,
            "manifest_filename": self.manifest_filename,
            "pin_memory": self.pin_memory,
            "shuffle_train": self.shuffle_train,
            "landmark_model_path": self.landmark_model_path,
            "letterbox_size": self.letterbox_size,
        }


DEFAULT_CONFIG = TrainConfig()


def get_default_config(num_classes: int = 2) -> TrainConfig:
    """Tạo đối tượng TrainConfig với số nhãn phân loại tùy chọn (2 hoặc 3 lớp)."""
    return TrainConfig(num_classes=num_classes)


def save_config(config: TrainConfig, path: str | Path) -> None:
    """Lưu đối tượng TrainConfig ra tệp JSON hoặc YAML."""
    config.save(path)


def load_config(path: str | Path) -> TrainConfig:
    """Nạp tệp cấu hình từ JSON hoặc YAML và trả về TrainConfig."""
    return TrainConfig.load(path)

In [ ]:
# Cell 4: Dataset và Bộ trích xuất 68 điểm mốc khuôn mặt MediaPipe (dataset.py)
# =====================================================================
# 1. BẢNG CHỈ SỐ ÁNH XẠ 68 ĐIỂM MỐC (MediaPipe 478 -> Dlib 68)
# Khớp chính xác với cấu trúc đồ thị FacialGraphBuilder trong block.py
# =====================================================================
LANDMARK_68_INDICES = [
    # Viền hàm (0-16): 17 điểm
    162, 234, 93, 58, 172, 136, 149, 148, 152, 377, 378, 365, 397, 288, 323, 454, 389,
    # Lông mày trái (17-21) & phải (22-26): 10 điểm
    71, 63, 105, 66, 107, 336, 296, 334, 293, 301,
    # Sống mũi & chân mũi (27-35): 9 điểm
    168, 197, 5, 4, 75, 97, 2, 326, 305,
    # Mắt trái (36-41) & Mắt phải (42-47): 12 điểm
    33, 160, 158, 133, 153, 144, 362, 385, 387, 263, 373, 380,
    # Môi ngoài (48-59) & Môi trong (60-67): 20 điểm
    61, 39, 37, 0, 267, 269, 291, 405, 314, 17, 84, 181,
    78, 82, 13, 312, 308, 317, 14, 87,
]

MEDIAPIPE_FACE_LANDMARKER_URL = (
    "https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task"
)


def letterbox(image: np.ndarray, new_size: int = 480, color: tuple = (114, 114, 114)) -> np.ndarray:
    """Resize ảnh giữ nguyên tỷ lệ khung hình với đệm màu viền."""
    h, w = image.shape[:2]
    scale = min(new_size / h, new_size / w)
    nw, nh = max(1, round(w * scale)), max(1, round(h * scale))
    resized = cv2.resize(image, (nw, nh), interpolation=cv2.INTER_LINEAR)
    canvas = np.full((new_size, new_size, 3), color, dtype=image.dtype)
    x, y = (new_size - nw) // 2, (new_size - nh) // 2
    canvas[y:y + nh, x:x + nw] = resized
    return canvas


# =====================================================================
# 2. BỘ TRÍCH XUẤT 68 ĐIỂM MỐC KHUÔN MẶT BẰNG MEDIAPIPE TASKS
# =====================================================================
class FaceLandmarkExtractor:
    """Trích xuất 68 điểm mốc chuẩn Dlib từ các khung hình ảnh bằng MediaPipe Tasks."""

    def __init__(self, model_path: Optional[str | Path] = None, time_interval: float = 0.0):
        if model_path is None:
            base_dir = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
            candidates = [
                base_dir.parent / "PhysicalBranch" / "models" / "face_landmarker.task",
                base_dir / "PhysicalBranch" / "models" / "face_landmarker.task",
                base_dir / "ai" / "PhysicalBranch" / "models" / "face_landmarker.task",
                base_dir / "face_landmarker.task",
            ]
            model_path = next((p for p in candidates if p.is_file()), candidates[-1])
        self.model_path = Path(model_path)
        self.time_interval = float(time_interval)
        self.landmarker = None

    def _init_landmarker(self):
        """Khởi tạo trễ (lazy init) để tương thích đa tiến trình của DataLoader & tự tải trên Kaggle/Colab."""
        if not self.model_path.is_file():
            self.model_path.parent.mkdir(parents=True, exist_ok=True)
            try:
                logger.info(f"Đang tải face_landmarker.task về {self.model_path}...")
                urllib.request.urlretrieve(MEDIAPIPE_FACE_LANDMARKER_URL, str(self.model_path))
            except Exception as e:
                raise FileNotFoundError(
                    f"Không tìm thấy file mô hình MediaPipe tại: {self.model_path} và không thể tự tải ({e})"
                )

        import mediapipe as mp
        from mediapipe.tasks.python import BaseOptions
        from mediapipe.tasks.python.vision import FaceLandmarker, FaceLandmarkerOptions, RunningMode

        options = FaceLandmarkerOptions(
            base_options=BaseOptions(model_asset_path=str(self.model_path)),
            running_mode=RunningMode.IMAGE,
            num_faces=1,
        )
        self.landmarker = FaceLandmarker.create_from_options(options)

    def extract(self, frames: List[np.ndarray]) -> np.ndarray:
        """
        Nhận danh sách frames BGR và trích xuất chuỗi 68 điểm mốc chuẩn hóa trong [0, 1].
        Trả về numpy array kích thước: (T, 68, 2)
        """
        if self.landmarker is None:
            self._init_landmarker()

        import mediapipe as mp

        landmarks_seq = []
        last_valid = None

        for frame in frames:
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
            result = self.landmarker.detect(mp_image)

            if result.face_landmarks:
                pts = result.face_landmarks[0]
                lm68 = np.array([[pts[i].x, pts[i].y] for i in LANDMARK_68_INDICES], dtype=np.float32)
                last_valid = lm68
            else:
                # Forward-fill: dùng lại landmark của frame hợp lệ gần nhất nếu bị mất mặt
                lm68 = last_valid.copy() if last_valid is not None else np.zeros((68, 2), dtype=np.float32)

            landmarks_seq.append(lm68)

        # Backward-fill: nếu các frame đầu chưa thấy mặt nhưng sau đó phát hiện được
        if last_valid is not None:
            for i in range(len(landmarks_seq)):
                if np.all(landmarks_seq[i] == 0):
                    landmarks_seq[i] = last_valid.copy()
                else:
                    break

        return np.array(landmarks_seq, dtype=np.float32)

    def extract_from_video(
        self,
        video_path: str | Path,
        time_interval: Optional[float] = None,
        window_sec: Optional[float] = None,
    ) -> Tuple[np.ndarray, List[np.ndarray], Dict[str, Any]]:
        """
        Đọc frame trực tiếp từ video file và trích xuất chuỗi 68 landmark.
        - time_interval: khoảng thời gian (giây) giữa 2 frame lấy mẫu.
                         Nếu == 0: đọc toàn bộ frame tuần tự thông thường từ video.
                         Nếu None: lấy giá trị self.time_interval.
        - window_sec: thời lượng cửa sổ (giây) cần lấy. None = toàn bộ video.
        """
        video_path = Path(video_path)
        cap = cv2.VideoCapture(str(video_path))
        if not cap.isOpened():
            raise IOError(f"Không thể mở video: {video_path}")

        fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        duration_sec = total_frames / fps if fps > 0 else 0.0

        ti = self.time_interval if time_interval is None else float(time_interval)
        if ti < 0:
            raise ValueError(f"time_interval phải là số không âm (>= 0): {ti}")

        if ti == 0:
            if window_sec is None:
                indices = np.arange(total_frames, dtype=int)
            else:
                win_sec = min(window_sec, duration_sec)
                num_frames = max(1, min(int(round(win_sec * fps)), total_frames))
                indices = np.arange(num_frames, dtype=int)
        else:
            if window_sec is None:
                timestamps = np.arange(0, duration_sec, ti)
            else:
                win_sec = min(window_sec, duration_sec)
                timestamps = np.arange(0, win_sec, ti)
            if len(timestamps) == 0:
                timestamps = np.array([0.0])
            indices = np.rint(timestamps * fps).astype(int)
            indices = np.clip(indices, 0, total_frames - 1)

        frames = []
        curr_frame = indices[0] if len(indices) > 0 else 0
        cap.set(cv2.CAP_PROP_POS_FRAMES, curr_frame)

        for target_idx in indices:
            while curr_frame < target_idx:
                if not cap.grab():
                    break
                curr_frame += 1
            ret, frame = cap.read()
            if not ret:
                break
            frames.append(frame)
            curr_frame += 1

        cap.release()

        if len(frames) < len(indices):
            last = frames[-1] if frames else np.zeros((480, 640, 3), dtype=np.uint8)
            frames.extend([last.copy()] * (len(indices) - len(frames)))

        lm68 = self.extract(frames)
        meta = {
            "video_path": str(video_path),
            "fps": fps,
            "total_frames": total_frames,
            "sampled_frames": len(frames),
            "duration_sec": duration_sec,
            "time_interval": ti,
        }
        return lm68, frames, meta


# =====================================================================
# 3. LỚP GCNDATASET
# =====================================================================
class GCNDataset(Dataset):
    """
    Dataset đọc video từ thư mục dữ liệu (vd: E:/LSTM/data_processed),
    hỗ trợ:
      - Từng video có FPS riêng (60, 30, 25 FPS, v.v.).
      - time_interval = 0 -> đọc frame tuần tự thông thường từ video (không bỏ cách frame).
      - time_interval > 0 -> cơ chế lấy mẫu sau mỗi time_interval giây (time-domain sampling).
      - window_sec = None -> lấy frame trên toàn bộ thời lượng video.
      - window_sec = số giây -> lấy frame trong phạm vi cửa sổ đó.
    """

    def __init__(
        self,
        dataset_dir: str | Path = r"E:\LSTM\data_processed",
        split: str = "train",
        time_interval: float = 0.2,
        window_sec: Optional[float] = 10.0,
        manifest_filename: str = "dataset_merged_split.csv",
        landmark_model_path: Optional[str | Path] = None,
        letterbox_size: int = 480,
        return_frames: bool = False,
    ):
        if split not in ("train", "val", "test"):
            raise ValueError(f"Split không hợp lệ: {split}. Chỉ chấp nhận 'train', 'val', 'test'.")

        self.manifest_filename = manifest_filename

        if time_interval < 0:
            raise ValueError(f"time_interval phải là số không âm (>= 0): {time_interval}")
        if window_sec is not None and window_sec <= 0:
            raise ValueError(f"window_sec phải là số dương hoặc None: {window_sec}")

        self.root = Path(dataset_dir)
        self.split = split
        self.time_interval = float(time_interval)
        self.window_sec = float(window_sec) if window_sec is not None else None
        self.letterbox_size = letterbox_size
        self.return_frames = return_frames
        self.extractor = FaceLandmarkExtractor(landmark_model_path, time_interval=self.time_interval)
        self.samples = self._load_samples()

        if len(self.samples) == 0:
            logging.warning(f"Không tìm thấy mẫu nào trong {self.root} cho split={split}")

    def _load_samples(self) -> List[Dict[str, Any]]:
        """Nạp danh sách video từ manifest CSV hoặc quét thư mục dự phòng."""
        manifest = self.root / self.manifest_filename
        samples = []

        if manifest.is_file():
            with manifest.open(encoding="utf-8-sig", newline="") as f:
                reader = csv.DictReader(f)
                for row in reader:
                    if row.get("split", "").strip().lower() == self.split:
                        rel_path = row.get("rel_path") or row.get("processed_path")
                        samples.append({
                            "path": self.root / rel_path.replace("\\", "/"),
                            "label": int(row["label"]),
                            "video_id": row.get("video_id") or row.get("clip_id") or Path(rel_path).stem,
                            "fps": float(row.get("fps", 0.0)),
                            "total_frames": int(float(row.get("frame_count", 0))),
                            "duration_sec": float(row.get("duration_sec", 0.0)),
                        })
        else:
            for label, folder in ((0, "0_alert"), (1, "1_drowsy")):
                split_dir = self.root / self.split / folder
                if split_dir.is_dir():
                    for p in sorted(split_dir.glob("*.mp4")):
                        samples.append({
                            "path": p,
                            "label": label,
                            "video_id": p.stem,
                            "fps": 0.0,
                            "total_frames": 0,
                            "duration_sec": 0.0,
                        })
        return samples

    def _get_frame_indices(self, total_frames: int, fps: float) -> np.ndarray:
        """Tính toán các chỉ số frame cần lấy dựa trên miền thời gian thực và FPS riêng của video."""
        if total_frames <= 0:
            return np.array([0], dtype=int)

        fps = fps if fps > 0 else 30.0
        duration_sec = total_frames / fps

        # TRƯỜNG HỢP 1: time_interval == 0 -> Đọc frame tuần tự thông thường từ video (không bỏ cách frame)
        if self.time_interval == 0:
            if self.window_sec is None:
                # Toàn bộ video tuần tự
                return np.arange(total_frames, dtype=int)
            else:
                # Chuỗi frame liên tiếp trong phạm vi cửa sổ window_sec
                win_sec = min(self.window_sec, duration_sec)
                num_frames = int(round(win_sec * fps))
                num_frames = max(1, min(num_frames, total_frames))
                max_start = max(0, total_frames - num_frames)
                start_idx = random.randint(0, max_start) if (self.split == "train" and max_start > 0) else max_start // 2
                return np.arange(start_idx, start_idx + num_frames, dtype=int)

        # TRƯỜNG HỢP 2: time_interval > 0 -> Lấy mẫu cách đều theo thời gian thực (time-domain sampling)
        if self.window_sec is None:
            # Lấy mẫu trên toàn bộ thời lượng video
            timestamps = np.arange(0, duration_sec, self.time_interval)
            if len(timestamps) == 0:
                timestamps = np.array([0.0])
        else:
            # Lấy mẫu trong phạm vi cửa sổ window_sec
            win = min(self.window_sec, duration_sec)
            max_start = max(0.0, duration_sec - win)
            start_sec = random.uniform(0.0, max_start) if (self.split == "train" and max_start > 0) else max_start / 2.0
            timestamps = start_sec + np.arange(0, win, self.time_interval)
            if len(timestamps) == 0:
                timestamps = np.array([start_sec])

        # Quy đổi từ mốc thời gian vật lý sang chỉ số frame theo FPS riêng của video
        indices = np.rint(timestamps * fps).astype(int)
        indices = np.clip(indices, 0, total_frames - 1)
        return indices

    def _read_frames(self, path: Path, indices: np.ndarray) -> List[np.ndarray]:
        """Đọc nhanh các frame tại các chỉ số được chỉ định bằng cv2.VideoCapture."""
        cap = cv2.VideoCapture(str(path))
        if not cap.isOpened():
            raise IOError(f"Không thể mở video: {path}")

        frames = []
        curr_frame = indices[0]
        cap.set(cv2.CAP_PROP_POS_FRAMES, curr_frame)

        for target_idx in indices:
            while curr_frame < target_idx:
                if not cap.grab():
                    break
                curr_frame += 1
            ret, frame = cap.read()
            if not ret:
                break
            frames.append(frame)
            curr_frame += 1

        cap.release()

        # Bù frame cuối nếu video kết thúc sớm hơn dự kiến
        if len(frames) < len(indices):
            last = frames[-1] if frames else np.zeros((480, 640, 3), dtype=np.uint8)
            frames.extend([last.copy()] * (len(indices) - len(frames)))

        return frames

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, index: int) -> Tuple[torch.Tensor, torch.Tensor, Dict[str, Any]]:
        sample = self.samples[index]
        fps = sample["fps"]
        total_frames = sample["total_frames"]

        # Nếu thiếu metadata, đọc trực tiếp từ video file
        if fps <= 0 or total_frames <= 0:
            cap = cv2.VideoCapture(str(sample["path"]))
            fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
            total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            cap.release()
            sample["fps"] = fps
            sample["total_frames"] = total_frames

        indices = self._get_frame_indices(total_frames, fps)
        frames = self._read_frames(sample["path"], indices)

        if self.letterbox_size:
            frames = [letterbox(f, new_size=self.letterbox_size) for f in frames]

        # Trích xuất 68 điểm mốc: shape (T, 68, 2)
        lm68 = self.extractor.extract(frames)

        # Chuyển đổi sang tensor chuẩn cho GCN: shape (2, T, 68)
        landmarks_tensor = torch.from_numpy(lm68).permute(2, 0, 1)

        label_tensor = torch.tensor(sample["label"], dtype=torch.long)
        meta = {
            "video_id": sample["video_id"],
            "path": str(sample["path"]),
            "fps": fps,
            "num_sampled_frames": len(indices),
            "duration_sec": total_frames / fps if fps > 0 else 0.0,
            "time_interval": self.time_interval,
        }

        if self.return_frames:
            return landmarks_tensor, label_tensor, meta, frames
        return landmarks_tensor, label_tensor, meta


FaceLandmarkDataset = GCNDataset
FaceLandmarkExtractorDataset = GCNDataset


# =====================================================================
# 4. HÀM VẼ LANDMARK VÀ HIỂN THỊ VIDEO
# =====================================================================
def get_landmark_region_color(idx: int) -> Tuple[int, int, int]:
    """
    Trả về màu sắc đại diện (BGR) theo từng vùng giải phẫu của hệ 68 điểm mốc:
      - 00 - 16: Viền hàm (Jawline) -> Xám bạc (200, 200, 200)
      - 17 - 21: Lông mày trái (Left Eyebrow) -> Vàng cam (255, 190, 0)
      - 22 - 26: Lông mày phải (Right Eyebrow) -> Vàng cam (255, 190, 0)
      - 27 - 30: Sống mũi (Nose Bridge) -> Vàng chanh (0, 255, 255)
      - 31 - 35: Chân mũi / Cánh mũi (Lower Nose) -> Vàng chanh (0, 255, 255)
      - 36 - 41: Mắt trái (Left Eye) -> Xanh lá (0, 255, 0)
      - 42 - 47: Mắt phải (Right Eye) -> Xanh lá (0, 255, 0)
      - 48 - 59: Môi ngoài (Outer Lip) -> Đỏ (0, 0, 255)
      - 60 - 67: Môi trong (Inner Lip) -> Cam đậm (0, 140, 255)
    """
    if 0 <= idx <= 16:
        return (200, 200, 200)  # Viền hàm
    elif 17 <= idx <= 26:
        return (255, 190, 0)    # Lông mày trái & phải
    elif 27 <= idx <= 35:
        return (0, 255, 255)    # Sống mũi & cánh mũi
    elif 36 <= idx <= 47:
        return (0, 255, 0)      # Mắt trái & phải
    elif 48 <= idx <= 59:
        return (0, 0, 255)      # Môi ngoài
    elif 60 <= idx <= 67:
        return (0, 140, 255)    # Môi trong
    return (0, 255, 0)


def draw_landmarks_on_frame(
    frame: np.ndarray,
    landmarks_xy: np.ndarray,
    show_ids: bool | List[int] | str = False,
    font_scale: float = 0.28,
    id_color: Optional[Tuple[int, int, int] | str] = (0, 255, 255),
    point_radius: int = 2,
    point_color: Optional[Tuple[int, int, int]] = (0, 255, 0),
    draw_contours: bool = True,
) -> np.ndarray:
    """
    Vẽ 68 điểm mốc khuôn mặt và các đường viền giải phẫu lên frame ảnh,
    đồng thời hỗ trợ hiển thị ID của các điểm theo hệ 68 điểm mốc của dự án (0-67).

    Tham số:
        frame: Ảnh gốc định dạng numpy array (H, W, 3) BGR.
        landmarks_xy: Tọa độ điểm mốc shape (68, 2) đã chuẩn hóa trong đoạn [0, 1].
        show_ids:
            - True: Hiển thị ID của toàn bộ 68 điểm (0 đến 67).
            - False / None: Tắt hiển thị ID.
            - List[int]: Danh sách các ID cụ thể cần hiển thị (vd: list(range(36, 48)) cho vùng mắt).
            - str: Từ khóa vùng giải phẫu: "eyes", "mouth", "nose", "eyebrows", "jaw", "all".
        font_scale: Kích thước chữ số hiển thị ID (mặc định 0.28, nhỏ gọn, rõ nét).
        id_color: Màu sắc chữ số (BGR). Mặc định vàng (0, 255, 255).
                  Nếu đặt id_color="region" hoặc id_color=None: Chữ số sẽ mang màu đồng bộ
                  theo từng vùng giải phẫu tương ứng.
        point_radius: Bán kính chấm tròn của mốc (mặc định 2).
        point_color: Màu sắc chấm tròn mốc (mặc định xanh lá). Nếu None: lấy theo màu vùng.
        draw_contours: Vẽ các đường liên kết giải phẫu (viền hàm, mắt, mày, môi, mũi).

    Hệ 68 điểm mốc của dự án:
        - 00 - 16: Viền hàm (17 điểm)
        - 17 - 21: Lông mày trái (5 điểm)
        - 22 - 26: Lông mày phải (5 điểm)
        - 27 - 30: Sống mũi (4 điểm)
        - 31 - 35: Chân / Cánh mũi (5 điểm)
        - 36 - 41: Mắt trái (6 điểm)
        - 42 - 47: Mắt phải (6 điểm)
        - 48 - 59: Môi ngoài (12 điểm)
        - 60 - 67: Môi trong (8 điểm)
    """
    vis = frame.copy()
    h, w = vis.shape[:2]

    num_pts = min(68, len(landmarks_xy))
    pts = []
    for i in range(num_pts):
        x = int(landmarks_xy[i, 0] * w)
        y = int(landmarks_xy[i, 1] * h)
        pts.append((x, y))

    # 1. Vẽ các đường nối giải phẫu (Contours)
    if draw_contours and len(pts) >= 68:
        def draw_chain(indices, closed=False, color=(0, 255, 255), thickness=1):
            for i in range(len(indices) - 1):
                cv2.line(vis, pts[indices[i]], pts[indices[i + 1]], color, thickness)
            if closed:
                cv2.line(vis, pts[indices[-1]], pts[indices[0]], color, thickness)

        draw_chain(list(range(0, 17)), closed=False, color=(200, 200, 200), thickness=1)  # Viền hàm
        draw_chain(list(range(17, 22)), closed=False, color=(255, 190, 0), thickness=1)   # Lông mày trái
        draw_chain(list(range(22, 27)), closed=False, color=(255, 190, 0), thickness=1)   # Lông mày phải
        draw_chain(list(range(27, 31)), closed=False, color=(0, 255, 255), thickness=1)   # Sống mũi
        draw_chain(list(range(31, 36)), closed=False, color=(0, 255, 255), thickness=1)   # Cánh mũi
        draw_chain(list(range(36, 42)), closed=True, color=(0, 255, 0), thickness=1)      # Mắt trái
        draw_chain(list(range(42, 48)), closed=True, color=(0, 255, 0), thickness=1)      # Mắt phải
        draw_chain(list(range(48, 60)), closed=True, color=(0, 0, 255), thickness=1)      # Môi ngoài
        draw_chain(list(range(60, 68)), closed=True, color=(0, 140, 255), thickness=1)    # Môi trong

    # 2. Vẽ 68 điểm mốc dạng chấm tròn
    for i, (x, y) in enumerate(pts):
        c = point_color if point_color is not None else get_landmark_region_color(i)
        cv2.circle(vis, (x, y), point_radius, c, -1)

    # 3. Vẽ ID của các điểm theo hệ 68 điểm của dự án (0 đến 67)
    if show_ids:
        # Xác định tập hợp ID cần vẽ
        if isinstance(show_ids, bool):
            target_ids = list(range(len(pts)))
        elif isinstance(show_ids, str):
            kw = show_ids.lower().strip()
            regions = {
                "jaw": list(range(0, 17)),
                "eyebrows": list(range(17, 27)),
                "left_eyebrow": list(range(17, 22)),
                "right_eyebrow": list(range(22, 27)),
                "nose": list(range(27, 36)),
                "nose_bridge": list(range(27, 31)),
                "lower_nose": list(range(31, 36)),
                "eyes": list(range(36, 48)),
                "left_eye": list(range(36, 42)),
                "right_eye": list(range(42, 48)),
                "mouth": list(range(48, 68)),
                "outer_lip": list(range(48, 60)),
                "inner_lip": list(range(60, 68)),
                "all": list(range(len(pts))),
            }
            target_ids = regions.get(kw, list(range(len(pts))))
        elif hasattr(show_ids, "__iter__"):
            target_ids = list(show_ids)
        else:
            target_ids = []

        for i in target_ids:
            if 0 <= i < len(pts):
                x, y = pts[i]
                # Canh chỉnh tọa độ đặt text ID để không bị che khuất điểm và không tràn mép ảnh
                tx = max(0, min(w - 18, x + 2))
                ty = max(10, min(h - 2, y - 2))
                text = str(i)

                # Xác định màu sắc text
                if id_color is None or id_color == "region":
                    text_color = get_landmark_region_color(i)
                elif isinstance(id_color, (tuple, list)):
                    text_color = tuple(id_color)
                else:
                    text_color = (0, 255, 255)

                # Viền chữ đen dày 2px tạo độ tương phản tuyệt đối trên mọi nền ảnh
                cv2.putText(vis, text, (tx, ty), cv2.FONT_HERSHEY_SIMPLEX, font_scale, (0, 0, 0), 2, cv2.LINE_AA)
                # Chữ số chính dày 1px
                cv2.putText(vis, text, (tx, ty), cv2.FONT_HERSHEY_SIMPLEX, font_scale, text_color, 1, cv2.LINE_AA)

    return vis


def visualize_frame_landmarks(
    frame: np.ndarray,
    landmarks_xy: np.ndarray,
    window_name: str = "Driver Guardian - 68 Facial Landmarks",
    show_ids: bool | List[int] | str = True,
    font_scale: float = 0.28,
    id_color: Optional[Tuple[int, int, int] | str] = (0, 255, 255),
    wait_key: bool = True,
) -> np.ndarray:
    """
    Hiển thị 1 frame ảnh kèm 68 điểm mốc và ID của chúng theo hệ 68 điểm của dự án.
    Rất thuận tiện cho việc kiểm tra nhanh và debug từng frame đơn lẻ.
    """
    vis = draw_landmarks_on_frame(
        frame=frame,
        landmarks_xy=landmarks_xy,
        show_ids=show_ids,
        font_scale=font_scale,
        id_color=id_color,
    )
    cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)
    h, w = vis.shape[:2]
    cv2.resizeWindow(window_name, min(w, 800), min(h, 600))
    cv2.imshow(window_name, vis)
    if wait_key:
        print(f"\n[Hiển thị Frame Landmark] Đang mở cửa sổ '{window_name}' (show_ids={show_ids}).")
        print(">>> Nhấn phím bất kỳ hoặc 'q' trên cửa sổ ảnh để đóng <<<")
        cv2.waitKey(0)
        cv2.destroyAllWindows()
    return vis


def visualize_video_landmarks(
    landmarks_tensor: torch.Tensor | np.ndarray,
    frames: List[np.ndarray],
    meta: Dict[str, Any],
    label: torch.Tensor | int = 0,
    time_interval: float = 0.2,
    window_name: str = "Driver Guardian - 68 Facial Landmarks",
    show_ids: bool | List[int] | str = False,
    font_scale: float = 0.28,
    id_color: Optional[Tuple[int, int, int] | str] = (0, 255, 255),
):
    """
    Hiển thị video tuần tự từng frame kèm các điểm mốc landmark đã trích xuất.
    Hỗ trợ hiển thị và bật/tắt ID của 68 điểm mốc theo hệ chuẩn của dự án (0-67).

    Phím điều khiển tương tác trên cửa sổ video:
        - [i] hoặc [I]: Bật / Tắt hiển thị ID của 68 điểm mốc trong thời gian thực.
        - [Space]: Tạm dừng / Tiếp tục phát video (dùng khi muốn dừng lại soi kỹ từng ID).
        - [d] / [Mũi tên phải]: Tiến 1 frame (khi đang tạm dừng).
        - [a] / [Mũi tên trái]: Lùi 1 frame (khi đang tạm dừng).
        - [q] hoặc [ESC]: Thoát video sớm.
    """
    # Xử lý linh hoạt landmarks_tensor: nhận cả (2, T, 68), (T, 68, 2), (B, 2, T, 68) hoặc np.ndarray
    if isinstance(landmarks_tensor, torch.Tensor):
        if landmarks_tensor.ndim == 4:
            landmarks_tensor = landmarks_tensor[0]
        if landmarks_tensor.shape[0] == 2:
            lm_np = landmarks_tensor.permute(1, 2, 0).cpu().numpy()
        else:
            lm_np = landmarks_tensor.cpu().numpy()
    elif isinstance(landmarks_tensor, np.ndarray):
        if landmarks_tensor.ndim == 3 and landmarks_tensor.shape[0] == 2:
            lm_np = np.transpose(landmarks_tensor, (1, 2, 0))
        else:
            lm_np = landmarks_tensor
    else:
        raise TypeError(f"landmarks_tensor không đúng kiểu: {type(landmarks_tensor)}")

    T = min(lm_np.shape[0], len(frames))
    fps = meta.get("fps", 30.0) or 30.0
    if time_interval == 0:
        delay_ms = max(1, int(1000 / fps))
        total_time_str = f"~{T / fps:.1f}s"
    else:
        delay_ms = max(1, int(time_interval * 1000))
        total_time_str = f"~{T * time_interval:.1f}s"

    lbl_val = label.item() if isinstance(label, torch.Tensor) else int(label)
    label_text = "ALERT (0)" if lbl_val == 0 else "DROWSY (1)"
    label_color = (0, 255, 0) if lbl_val == 0 else (0, 0, 255)

    cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)
    h, w = frames[0].shape[:2]
    cv2.resizeWindow(window_name, min(w, 800), min(h, 600))

    video_id = meta.get("video_id", "Sample")
    print(f"\n[Playing Video] {video_id} ({T} frames, {total_time_str})")
    print(">>> Phím nóng: [i] Bật/tắt ID (0-67) | [Space] Tạm dừng/Soi chi tiết | [a/d] Frame lùi/tiến | [q/ESC] Thoát <<<")

    t = 0
    paused = False
    show_ids_state = show_ids

    while t < T:
        vis = draw_landmarks_on_frame(
            frame=frames[t],
            landmarks_xy=lm_np[t],
            show_ids=show_ids_state,
            font_scale=font_scale,
            id_color=id_color,
        )

        # Dòng 1: Video ID & Trạng thái Alert / Drowsy
        cv2.putText(vis, f"Video: {video_id} | Status: {label_text}", (15, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 0, 0), 3, cv2.LINE_AA)
        cv2.putText(vis, f"Video: {video_id} | Status: {label_text}", (15, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.65, label_color, 2, cv2.LINE_AA)

        # Dòng 2: Chỉ số frame, mốc thời gian thực, FPS và trạng thái tạm dừng
        curr_time_sec = (t / fps) if time_interval == 0 else (t * time_interval)
        time_info = f"Frame: {t + 1}/{T} | Time: {curr_time_sec:.1f}s | FPS: {fps:.1f}"
        if paused:
            time_info += "  [PAUSED - Soi ID]"
        cv2.putText(vis, time_info, (15, 55),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 3, cv2.LINE_AA)
        cv2.putText(vis, time_info, (15, 55),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1, cv2.LINE_AA)

        # Dòng 3: Hướng dẫn phím nóng tương tác
        id_str = "ON" if show_ids_state else "OFF"
        guide = f"[i]: IDs ({id_str}) | [Space]: {'Resume' if paused else 'Pause'} | [q/ESC]: Exit"
        cv2.putText(vis, guide, (15, 80),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 0, 0), 3, cv2.LINE_AA)
        cv2.putText(vis, guide, (15, 80),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 255, 255), 1, cv2.LINE_AA)

        cv2.imshow(window_name, vis)

        # Kiểm tra nếu cửa sổ bị tắt bởi nút X
        if cv2.getWindowProperty(window_name, cv2.WND_PROP_VISIBLE) < 1:
            print("Cửa sổ đã được đóng.")
            break

        wait_ms = 0 if paused else delay_ms
        key = cv2.waitKey(wait_ms) & 0xFF

        if key in (ord('q'), ord('Q'), 27):
            print("Đã nhận lệnh dừng từ người dùng.")
            break
        elif key in (ord('i'), ord('I')):
            show_ids_state = not show_ids_state
            print(f">>> Landmark IDs (0-67): {'BẬT' if show_ids_state else 'TẮT'}")
            continue
        elif key == 32:  # Phím Space
            paused = not paused
            print(f">>> {'TẠM DỪNG' if paused else 'TIẾP TỤC'}")
            continue
        elif paused and key in (ord('d'), ord('D'), 83):  # Frame kế tiếp
            t = min(T - 1, t + 1)
            continue
        elif paused and key in (ord('a'), ord('A'), 81):  # Frame trước đó
            t = max(0, t - 1)
            continue

        if not paused:
            t += 1

    cv2.destroyAllWindows()


# =====================================================================
# 5. HÀM COLLATE VÀ BUILD DATALOADER
# =====================================================================
# =====================================================================
def collate_gcn(batch: List[Tuple[Any, ...]]):
    """
    Gom các mẫu vào batch.
    Khi window_sec=None (số frame T giữa các video có thể khác nhau) hoặc các video có độ dài khác nhau,
    tự động pad thêm frame 0 ở chiều thời gian T về max_T của batch và trả về tensor lengths.

    Returns:
        - padded_landmarks: Tensor (B, 2, max_T, 68)
        - labels          : LongTensor (B,)
        - lengths         : LongTensor (B,) chứa số frame thực tế trước khi pad của từng mẫu
        - metas           : List[Dict[str, Any]]
        - frames (tùy chọn): List[List[np.ndarray]] nếu dataset bật return_frames=True
    """
    has_frames = len(batch[0]) == 4
    if has_frames:
        landmarks, labels, metas, frames_list = zip(*batch)
    else:
        landmarks, labels, metas = zip(*batch)

    lengths = [lm.shape[1] for lm in landmarks]
    max_len = max(lengths)

    padded_landmarks = []
    updated_metas = []
    for lm, meta, t in zip(landmarks, metas, lengths):
        if t < max_len:
            pad_tensor = torch.zeros((lm.shape[0], max_len, lm.shape[2]), dtype=lm.dtype)
            pad_tensor[:, :t, :] = lm
            padded_landmarks.append(pad_tensor)
        else:
            padded_landmarks.append(lm)

        meta_copy = dict(meta)
        meta_copy["valid_frames"] = t
        meta_copy["padded_frames"] = max_len
        updated_metas.append(meta_copy)

    lengths_tensor = torch.tensor(lengths, dtype=torch.long)

    if has_frames:
        return torch.stack(padded_landmarks), torch.stack(labels), lengths_tensor, updated_metas, list(frames_list)
    return torch.stack(padded_landmarks), torch.stack(labels), lengths_tensor, updated_metas


def build_gcn_dataloaders(
    dataset_dir: str | Path = r"E:\LSTM\data_processed",
    batch_size: int = 16,
    num_workers: int = 0,
    time_interval: float = 0.2,
    window_sec: Optional[float] = 10.0,
    manifest_filename: str = "dataset_merged_split.csv",
    pin_memory: bool = True,
    shuffle_train: bool = True,
    landmark_model_path: Optional[str | Path] = None,
    letterbox_size: int = 480,
):
    """
    Xây dựng nhanh bộ DataLoader cho cả 3 tập train, val, test.
    - time_interval = 0: đọc toàn bộ frame tuần tự thông thường từ video.
    - time_interval > 0: lấy mẫu cách nhau time_interval giây (vd 0.2s tương đương 5 FPS).
    """
    train_ds = GCNDataset(
        dataset_dir=dataset_dir,
        split="train",
        time_interval=time_interval,
        window_sec=window_sec,
        manifest_filename=manifest_filename,
        landmark_model_path=landmark_model_path,
        letterbox_size=letterbox_size,
    )
    val_ds = GCNDataset(
        dataset_dir=dataset_dir,
        split="val",
        time_interval=time_interval,
        window_sec=window_sec,
        manifest_filename=manifest_filename,
        landmark_model_path=landmark_model_path,
        letterbox_size=letterbox_size,
    )
    test_ds = GCNDataset(
        dataset_dir=dataset_dir,
        split="test",
        time_interval=time_interval,
        window_sec=window_sec,
        manifest_filename=manifest_filename,
        landmark_model_path=landmark_model_path,
        letterbox_size=letterbox_size,
    )

    train_loader = DataLoader(
        train_ds, batch_size=batch_size, shuffle=shuffle_train, num_workers=num_workers,
        pin_memory=pin_memory, collate_fn=collate_gcn
    )
    val_loader = DataLoader(
        val_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers,
        pin_memory=pin_memory, collate_fn=collate_gcn
    )
    test_loader = DataLoader(
        test_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers,
        pin_memory=pin_memory, collate_fn=collate_gcn
    )
    return train_loader, val_loader, test_loader


In [ ]:
# Cell 5: Các khối xây dựng Đồ thị Không - Thời gian (block.py)
class FacialGraphBuilder:
    """
    Xây dựng cấu trúc đồ thị 68 điểm mốc khuôn mặt bằng torch_geometric.utils.
    Hỗ trợ xuất cả edge_index (Sparse) và Adjacency Matrix (Dense).
    """

    @staticmethod
    def get_edge_index(
        num_nodes: int = 68,
        vertical_pairs: Optional[List[Tuple[int, int]] | List[List[int]]] = None,
        cross_region_pairs: Optional[List[Tuple[int, int]] | List[List[int]]] = None,
    ) -> torch.Tensor:
        edges = []

        def add_chain(indices, closed=False):
            for i in range(len(indices) - 1):
                edges.append([indices[i], indices[i + 1]])
            if closed:
                edges.append([indices[-1], indices[0]])

        # 1. Cạnh giải phẫu cơ mặt (Anatomical Contours)
        add_chain(range(0, 17), closed=False)  # Viền hàm
        add_chain(range(17, 22), closed=False)  # Lông mày trái
        add_chain(range(22, 27), closed=False)  # Lông mày phải
        add_chain(range(27, 31), closed=False)  # Sống mũi
        add_chain(range(31, 36), closed=False)  # Cánh mũi
        add_chain(range(36, 42), closed=True)  # Mắt trái
        add_chain(range(42, 48), closed=True)  # Mắt phải
        add_chain(range(48, 60), closed=True)  # Môi ngoài
        add_chain(range(60, 68), closed=True)  # Môi trong

        # 2. Cạnh dọc đo độ mở Mắt (EAR) & Miệng (MAR)
        if vertical_pairs is None:
            vertical_pairs = [
                [37, 41], [38, 40], [43, 47], [44, 46],  # Mí mắt trên - dưới
                [50, 58], [51, 57], [52, 56], [61, 67], [62, 66], [63, 65]  # Môi trên - dưới
            ]

        # 3. Cạnh tiên nghiệm liên vùng (Mắt <-> Miệng, Mũi <-> Cằm)
        if cross_region_pairs is None:
            cross_region_pairs = [
                [37, 48], [41, 48], [38, 62], [40, 66],  # Mắt trái <-> Miệng
                [44, 54], [46, 54], [43, 62], [47, 66],  # Mắt phải <-> Miệng
                [30, 8], [30, 36], [30, 45], [27, 8]  # Chóp mũi <-> Cằm & Đuôi mắt
            ]

        all_edges = edges + [list(p) for p in vertical_pairs] + [list(p) for p in cross_region_pairs]
        edge_index = torch.tensor(all_edges, dtype=torch.long).t().contiguous()

        # Sử dụng hàm có sẵn của PyTorch Geometric để tạo đồ thị vô hướng và thêm self-loops
        edge_index = to_undirected(edge_index, num_nodes=num_nodes)
        edge_index, _ = add_self_loops(edge_index, num_nodes=num_nodes)
        return edge_index

    @classmethod
    def get_dense_adj(
        cls,
        num_nodes: int = 68,
        vertical_pairs: Optional[List[Tuple[int, int]] | List[List[int]]] = None,
        cross_region_pairs: Optional[List[Tuple[int, int]] | List[List[int]]] = None,
    ) -> torch.Tensor:
        """Trả về ma trận kề Dense kích thước (V, V) từ edge_index."""
        edge_index = cls.get_edge_index(
            num_nodes=num_nodes,
            vertical_pairs=vertical_pairs,
            cross_region_pairs=cross_region_pairs,
        )
        return to_dense_adj(edge_index, max_num_nodes=num_nodes).squeeze(0)


class KinematicFeatureExtractor(nn.Module):
    """
    Khối tiền xử lý chuẩn hóa tọa độ 68 điểm mốc:
    1. Đưa tọa độ tĩnh (x, y) về Hệ tọa độ Cục bộ Tâm mũi (điểm 30) & chia cho khoảng cách 2 mắt (36 <-> 45).
    2. Tính Vận tốc (dx, dy) và Gia tốc (d2x, d2y) đã chuẩn hóa theo kích thước khuôn mặt
       (giữ nguyên được quỹ đạo rơi của đầu khi tài xế ngủ gật).
    Hỗ trợ temporal mask để triệt tiêu cú sốc số học tại biên zero-padding khi batch có độ dài thay đổi.
    """
    def __init__(
        self,
        nose_idx: int = 30,
        left_eye_idx: int = 36,
        right_eye_idx: int = 45,
        eps: float = 1e-6,
    ):
        super().__init__()
        self.nose_idx = nose_idx
        self.left_eye_idx = left_eye_idx
        self.right_eye_idx = right_eye_idx
        self.eps = eps

    @staticmethod
    def build_temporal_mask(
        x: torch.Tensor,
        lengths: Optional[torch.Tensor] = None,
        eps: float = 1e-6,
    ) -> torch.Tensor:
        """
        Tạo binary float mask kích thước (B, 1, T, 1) biểu diễn các frame hợp lệ (1.0) và frame đệm (0.0).
        - Nếu lengths được truyền vào: tạo mask chính xác từ lengths.
        - Nếu lengths is None: tự động nhận diện từ biên độ tọa độ (frame hợp lệ có tọa độ khác 0).
        """
        B, _, T, _ = x.shape
        device = x.device
        if lengths is not None:
            t_idx = torch.arange(T, device=device).unsqueeze(0)  # (1, T)
            mask = (t_idx < lengths.to(device).unsqueeze(1)).float()  # (B, T)
            return mask.view(B, 1, T, 1)
        else:
            # Tự nhận diện frame hợp lệ (tổng tọa độ x, y khác 0)
            frame_activity = x[:, :2].abs().sum(dim=(1, 3), keepdim=True)  # (B, 1, T, 1)
            return (frame_activity > eps).float()

    def forward(self, x: torch.Tensor, mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        """
        Input : (B, 2, T, V=68) - Tọa độ pixel hoặc [0, 1] trong Hệ tọa độ Khung ảnh.
        mask  : (B, 1, T, 1)    - Float mask (1.0 = valid, 0.0 = padded)
        Output: (B, 6, T, V=68) - [x_norm, y_norm, dx_norm, dy_norm, d2x_norm, d2y_norm]
        """
        if x.size(1) == 6:
            return x * mask if mask is not None else x

        # 1. Tính khoảng cách giải phẫu chuẩn d_t (Inter-Ocular Distance giữa điểm 36 và 45)
        # eye_dist shape: (B, 1, T, 1)
        p_left_eye = x[:, :, :, self.left_eye_idx : self.left_eye_idx + 1]   # (B, 2, T, 1)
        p_right_eye = x[:, :, :, self.right_eye_idx : self.right_eye_idx + 1] # (B, 2, T, 1)
        raw_iod = torch.norm(p_right_eye - p_left_eye, p=2, dim=1, keepdim=True)
        # Tránh chia cho eps=1e-6 tại các frame pad gây spike số học khổng lồ
        iod_scale = raw_iod.clamp(min=self.eps)

        # 2. Chuẩn hóa tọa độ hình thái tĩnh theo Tâm mũi (điểm 30) và tỷ lệ IOD
        nose_pos = x[:, :, :, self.nose_idx : self.nose_idx + 1]              # (B, 2, T, 1)
        pos_norm = (x - nose_pos) / iod_scale                                 # (B, 2, T, V)

        # 3. Tính Vận tốc (dx, dy) và Gia tốc (d2x, d2y) chuẩn hóa theo IOD
        # (Tính diff trên x trước khi trừ mũi để giữ được vận tốc gật đầu của cả khuôn mặt)
        vel_raw = torch.diff(x, n=1, dim=2)
        vel_raw = F.pad(vel_raw, (0, 0, 1, 0), mode='replicate')

        if mask is not None:
            # Triệt tiêu bước nhảy biên giữa frame hợp lệ cuối cùng và frame pad đầu tiên
            pair_valid_vel = mask[:, :, 1:, :] * mask[:, :, :-1, :]
            pair_valid_vel = F.pad(pair_valid_vel, (0, 0, 1, 0), mode='constant', value=1.0)
            vel_raw = vel_raw * pair_valid_vel

        vel_norm = vel_raw / iod_scale                                        # (B, 2, T, V)

        acc_raw = torch.diff(vel_norm, n=1, dim=2)
        acc_raw = F.pad(acc_raw, (0, 0, 1, 0), mode='replicate')

        if mask is not None:
            pair_valid_acc = mask[:, :, 1:, :] * mask[:, :, :-1, :]
            pair_valid_acc = F.pad(pair_valid_acc, (0, 0, 1, 0), mode='constant', value=1.0)
            acc_raw = acc_raw * pair_valid_acc

        acc_norm = acc_raw                                                    # (B, 2, T, V)

        # Ghép thành 6 kênh đặc trưng chuẩn hóa hoàn chỉnh
        out = torch.cat([pos_norm, vel_norm, acc_norm], dim=1)               # (B, 6, T, V)

        if mask is not None:
            out = out * mask

        return out


class PyGSpatialGraphConv(nn.Module):
    """
    Khối tích chập không gian tận dụng DenseGCNConv và DenseGATConv của PyTorch Geometric.
    - DenseGCNConv: Học trên ma trận tiên nghiệm A_prior + ma trận tham số tự học A_adaptive.
    - DenseGATConv: Tự động tính Attention động (A_dynamic) giữa các điểm mốc theo từng mẫu.
    """

    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        adj_prior: torch.Tensor,
        gat_heads: int = 4,
        dropout: float = 0.1,
        adaptive_init_scale: float = 1e-5,
    ):
        super().__init__()
        num_nodes = adj_prior.size(0)
        self.register_buffer('adj_prior', adj_prior)

        # Ma trận kề tự học toàn cục A_adaptive
        self.adj_adaptive = nn.Parameter(torch.randn(num_nodes, num_nodes) * adaptive_init_scale)

        # 1. Nhánh GCN tĩnh + thích ứng (PyG DenseGCNConv tự động chuẩn hóa D^-1/2 A D^-1/2)
        self.gcn = DenseGCNConv(in_channels, out_channels, improved=False, bias=True)

        # 2. Nhánh Graph Attention động (PyG DenseGATConv)
        self.gat = DenseGATConv(
            in_channels,
            out_channels,
            heads=gat_heads,
            concat=False,
            dropout=dropout,
            bias=True
        )
        self.alpha = nn.Parameter(torch.zeros(1))  # Hệ số hòa trộn nhánh GAT động

        self.bn = nn.BatchNorm2d(out_channels)
        self.act = nn.GELU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x shape: (B, C_in, T, V) -> Output shape: (B, C_out, T, V)
        """
        B, C, T, V = x.shape

        # Chuyển chiều về dạng chuẩn của PyG Dense layers: (B * T, V, C_in)
        x_dense = x.permute(0, 2, 3, 1).contiguous().view(B * T, V, C)

        # Tổng hợp ma trận kề cho nhánh GCN
        adj_total = self.adj_prior + self.adj_adaptive

        # Thực thi DenseGCNConv và DenseGATConv (add_loop=False vì adj_prior đã có self-loops)
        out_gcn = self.gcn(x_dense, adj_total, add_loop=False)  # (B*T, V, C_out)
        out_gat = self.gat(x_dense, self.adj_prior, add_loop=False)  # (B*T, V, C_out)

        out = out_gcn + self.alpha * out_gat  # (B*T, V, C_out)

        # Khôi phục lại chiều (B, C_out, T, V)
        out = out.view(B, T, V, -1).permute(0, 3, 1, 2).contiguous()
        return self.act(self.bn(out))


class MultiScaleTemporalConv(nn.Module):
    """
    Khối Tích chập Thời gian Đa tỷ lệ (Multi-Scale TCN) sử dụng nn.Conv2d với Dilation khác nhau:
    - Nhánh 1 (Dilation = 1): Bắt chuyển động nhanh (chớp mắt).
    - Nhánh 2 (Dilation = 2): Bắt chuyển động kéo dài (ngáp, ngủ gật).
    """

    def __init__(
        self,
        channels: int,
        kernel_size: int = 5,
        stride: int = 1,
        dropout: float = 0.2,
        dilations: Tuple[int, int] = (1, 2),
    ):
        super().__init__()
        mid_channels = channels // 2
        d1, d2 = dilations
        pad_d1 = (kernel_size - 1) * d1 // 2
        pad_d2 = (kernel_size - 1) * d2 // 2

        self.branch1 = nn.Conv2d(
            channels, mid_channels,
            kernel_size=(kernel_size, 1),
            stride=(stride, 1),
            padding=(pad_d1, 0),
            dilation=(d1, 1)
        )
        self.branch2 = nn.Conv2d(
            channels, channels - mid_channels,
            kernel_size=(kernel_size, 1),
            stride=(stride, 1),
            padding=(pad_d2, 0),
            dilation=(d2, 1)
        )

        self.bn = nn.BatchNorm2d(channels)
        self.drop = nn.Dropout(dropout, inplace=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = torch.cat([self.branch1(x), self.branch2(x)], dim=1)
        return self.drop(self.bn(out))


class STGCNBlock(nn.Module):
    """
    Khối ST-GCN hoàn chỉnh: PyGSpatialGraphConv -> MultiScaleTemporalConv -> Residual Connection.
    Hỗ trợ temporal mask để triệt tiêu bias rò rỉ tại các frame zero-padding.
    """

    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        adj_prior: torch.Tensor,
        temporal_kernel: int = 5,
        stride: int = 1,
        dropout: float = 0.2,
        gat_heads: int = 4,
        adaptive_init_scale: float = 1e-5,
        dilations: Tuple[int, int] = (1, 2),
    ):
        super().__init__()
        self.stride = stride
        self.sgcn = PyGSpatialGraphConv(
            in_channels,
            out_channels,
            adj_prior,
            gat_heads=gat_heads,
            dropout=dropout,
            adaptive_init_scale=adaptive_init_scale,
        )
        self.tgcn = MultiScaleTemporalConv(
            out_channels,
            kernel_size=temporal_kernel,
            stride=stride,
            dropout=dropout,
            dilations=dilations,
        )

        if in_channels == out_channels and stride == 1:
            self.residual = nn.Identity()
        else:
            self.residual = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=(stride, 1), bias=False),
                nn.BatchNorm2d(out_channels)
            )

        self.act = nn.GELU()

    def forward(self, x: torch.Tensor, mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        """
        x   : (B, C_in, T_in, V)
        mask: Optional (B, 1, T_in, 1) - Float mask (1.0 = valid, 0.0 = padded)
        """
        res = self.residual(x)
        x = self.sgcn(x)
        if mask is not None:
            # Triệt tiêu bias của GCN/GAT/BN ở vùng pad trước khi tích chập thời gian
            x = x * mask
        x = self.tgcn(x)
        out = self.act(x + res)
        if mask is not None:
            mask_out = mask[:, :, ::self.stride, :][:, :, :out.size(2), :]
            out = out * mask_out
        return out

In [ ]:
# Cell 6: Kiến trúc mô hình DriverSTGCN và DriverSTGCNTransformer (model.py)
class DriverSTGCN(nn.Module):
    """
    Model 1: ST-GCN Tiêu chuẩn siêu nhẹ (Tối ưu cho Edge AI / Suy luận thời gian thực).
    Ghép nối chuỗi STGCNBlock + Global Adaptive Pooling + MLP Classifier.
    Khởi tạo thuần túy qua các tham số thông thường, hoàn toàn độc lập với đối tượng config.
    """
    def __init__(
        self,
        in_channels: int = 6,
        num_classes: int = 2,
        num_nodes: int = 68,
        hidden_dims: Optional[List[int]] = None,
        strides: Optional[List[int]] = None,
        dropout: float = 0.25,
        temporal_kernel: int = 5,
        gat_heads: int = 4,
        classifier_hidden_dim: Optional[int] = None,
        nose_idx: int = 30,
        left_eye_idx: int = 36,
        right_eye_idx: int = 45,
        eps: float = 1e-6,
    ):
        super().__init__()
        if hidden_dims is None:
            hidden_dims = [64, 128, 256]
        if strides is None:
            strides = [1, 2, 2]

        assert len(hidden_dims) == len(strides), "hidden_dims và strides phải có cùng độ dài."

        # 1. Khởi tạo ma trận kề tiên nghiệm từ block.py
        adj_prior = FacialGraphBuilder.get_dense_adj(num_nodes=num_nodes)

        # 2. Khối trích xuất động học & Chuẩn hóa BatchNorm đầu vào trên từng điểm mốc
        self.kinematics = KinematicFeatureExtractor(
            nose_idx=nose_idx,
            left_eye_idx=left_eye_idx,
            right_eye_idx=right_eye_idx,
            eps=eps,
        )
        self.input_bn = nn.BatchNorm1d(in_channels * num_nodes)

        # 3. Chuỗi các khối ST-GCN (dùng nn.ModuleList)
        blocks = []
        curr_dim = in_channels
        for out_dim, stride in zip(hidden_dims, strides):
            blocks.append(
                STGCNBlock(
                    in_channels=curr_dim,
                    out_channels=out_dim,
                    adj_prior=adj_prior,
                    temporal_kernel=temporal_kernel,
                    stride=stride,
                    dropout=dropout,
                    gat_heads=gat_heads,
                )
            )
            curr_dim = out_dim
        self.backbone = nn.ModuleList(blocks)

        # 4. Đầu phân loại (Classification Head)
        cls_mid = classifier_hidden_dim if classifier_hidden_dim is not None else (curr_dim // 2)
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.LayerNorm(curr_dim),
            nn.Dropout(dropout),
            nn.Linear(curr_dim, cls_mid),
            nn.GELU(),
            nn.Linear(cls_mid, num_classes)
        )

    def _forward_features_with_mask(
        self,
        x: torch.Tensor,
        lengths: Optional[torch.Tensor] = None,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Trích xuất đặc trưng kèm theo mask thời gian đã downsample tương ứng.
        Trả về (feat, mask) trong đó:
          - feat: (B, C_last, T_out, V)
          - mask: (B, 1, T_out, 1)
        """
        mask = self.kinematics.build_temporal_mask(x, lengths=lengths)
        x = self.kinematics(x, mask=mask)  # Đảm bảo đầu vào có đủ 6 kênh [x, y, dx, dy, d2x, d2y]
        B, C, T, V = x.shape

        # Chuẩn hóa đầu vào dọc theo (C * V)
        x = x.permute(0, 1, 3, 2).contiguous().view(B, C * V, T)
        x = self.input_bn(x)
        x = x.view(B, C, V, T).permute(0, 1, 3, 2).contiguous()
        x = x * mask  # Triệt tiêu bias của BatchNorm tại các frame zero-padding

        curr_mask = mask
        for block in self.backbone:
            x = block(x, mask=curr_mask)
            curr_mask = curr_mask[:, :, ::block.stride, :][:, :, :x.size(2), :]

        return x, curr_mask

    def forward_features(
        self,
        x: torch.Tensor,
        lengths: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        """Trích xuất đặc trưng không gian - thời gian trước khi Pooling."""
        feat, _ = self._forward_features_with_mask(x, lengths=lengths)
        return feat

    def forward(
        self,
        x: torch.Tensor,
        lengths: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        """
        x      : (B, 2, T, V) hoặc (B, 6, T, V)
        lengths: LongTensor (B,) [Tùy chọn - số frame hợp lệ trước khi pad của từng mẫu]
        Output : (B, num_classes)
        """
        feat, mask = self._forward_features_with_mask(x, lengths=lengths)  # (B, C_last, T_out, V), (B, 1, T_out, 1)

        # Masked Global Average Pooling:
        # 1. Trung bình trên 68 đỉnh không gian
        feat_spatial = feat.mean(dim=-1, keepdim=True)                     # (B, C_last, T_out, 1)
        # 2. Trung bình trên đúng số lượng frame hợp lệ L_out (loại bỏ hoàn toàn ảnh hưởng của frame pad)
        valid_t = mask.sum(dim=2, keepdim=True).clamp(min=1.0)             # (B, 1, 1, 1)
        pooled = (feat_spatial * mask).sum(dim=2, keepdim=True) / valid_t  # (B, C_last, 1, 1)

        return self.classifier(pooled)                                     # (B, num_classes)


class DriverSTGCNTransformer(nn.Module):
    """
    Model 2: Kiến trúc Lai ST-GCN + PyTorch TransformerEncoder.
    Dùng ST-GCN để học tương quan đồ thị cơ mặt cục bộ, sau đó đưa qua
    nn.TransformerEncoder để học phụ thuộc chuỗi thời gian dài hạn.
    Khởi tạo thuần túy qua các tham số thông thường, hoàn toàn độc lập với đối tượng config.
    """
    def __init__(
        self,
        in_channels: int = 6,
        num_classes: int = 2,
        num_nodes: int = 68,
        d_model: int = 128,
        nhead: int = 4,
        num_transformer_layers: int = 2,
        max_len: int = 300,
        dropout: float = 0.2,
        stem_channels: Optional[List[int]] = None,
        stem_strides: Optional[List[int]] = None,
        dim_feedforward: Optional[int] = None,
        init_std: float = 0.02,
        nose_idx: int = 30,
        left_eye_idx: int = 36,
        right_eye_idx: int = 45,
        eps: float = 1e-6,
    ):
        super().__init__()
        if stem_channels is None:
            stem_channels = [64, d_model]
        if stem_strides is None:
            stem_strides = [1, 2]
        if dim_feedforward is None:
            dim_feedforward = d_model * 2

        adj_prior = FacialGraphBuilder.get_dense_adj(num_nodes=num_nodes)

        self.kinematics = KinematicFeatureExtractor(
            nose_idx=nose_idx,
            left_eye_idx=left_eye_idx,
            right_eye_idx=right_eye_idx,
            eps=eps,
        )
        self.input_bn = nn.BatchNorm1d(in_channels * num_nodes)

        # 1. Tầng ST-GCN Stem rút trích đặc trưng đồ thị khuôn mặt
        self.stgcn_stem = nn.ModuleList([
            STGCNBlock(in_channels, stem_channels[0], adj_prior, stride=stem_strides[0], dropout=dropout),
            STGCNBlock(stem_channels[0], stem_channels[1], adj_prior, stride=stem_strides[1], dropout=dropout)
        ])

        # 2. Trọng số chú ý không gian (Spatial Node Attention Pooling) gộp V=68 đỉnh về 1 vector/frame
        self.node_attention = nn.Sequential(
            nn.Conv1d(stem_channels[-1], 1, kernel_size=1),
            nn.Softmax(dim=-1)
        )

        # 3. Khối TransformerEncoder chuẩn của PyTorch dọc trục thời gian T
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model) * init_std)
        self.pos_embedding = nn.Parameter(torch.randn(1, max_len + 1, d_model) * init_std)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation='gelu',
            batch_first=True,
            norm_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_transformer_layers)

        # 4. Đầu dự đoán trạng thái buồn ngủ
        self.head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, num_classes)
        )

    def forward(
        self,
        x: torch.Tensor,
        lengths: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        """
        x      : (B, 2, T, V) hoặc (B, 6, T, V)
        lengths: LongTensor (B,) [Tùy chọn - số frame hợp lệ trước khi pad]
        Output : (B, num_classes)
        """
        mask = self.kinematics.build_temporal_mask(x, lengths=lengths)
        x = self.kinematics(x, mask=mask)
        B, C, T, V = x.shape

        x = x.permute(0, 1, 3, 2).contiguous().view(B, C * V, T)
        x = self.input_bn(x)
        x = x.view(B, C, V, T).permute(0, 1, 3, 2).contiguous()
        x = x * mask

        # Qua ST-GCN Stem:
        curr_mask = mask
        for block in self.stgcn_stem:
            x = block(x, mask=curr_mask)
            curr_mask = curr_mask[:, :, ::block.stride, :][:, :, :x.size(2), :]

        _, D, T_prime, _ = x.shape

        # Gộp 68 đỉnh bằng Node Attention cho mỗi bước thời gian
        x_flat = x.permute(0, 2, 1, 3).contiguous().view(B * T_prime, D, V)
        attn_weights = self.node_attention(x_flat)                  # (B*T', 1, V)
        x_spatial_pooled = torch.sum(x_flat * attn_weights, dim=-1) # (B*T', D)
        seq = x_spatial_pooled.view(B, T_prime, D)                  # (B, T', D)

        # Ghép [CLS] token và Positional Embedding rồi đưa qua TransformerEncoder
        cls_tokens = self.cls_token.expand(B, -1, -1)               # (B, 1, D)
        seq = torch.cat([cls_tokens, seq], dim=1)                   # (B, T'+1, D)

        seq_len = seq.size(1)
        if seq_len <= self.pos_embedding.size(1):
            pos_emb = self.pos_embedding[:, :seq_len, :]
        else:
            pos_emb = F.interpolate(
                self.pos_embedding.permute(0, 2, 1),
                size=seq_len,
                mode='linear',
                align_corners=False
            ).permute(0, 2, 1)

        seq = seq + pos_emb

        # Xây dựng padding mask cho Transformer: True = bỏ qua (vùng pad), False = hợp lệ
        padding_mask = torch.zeros((B, seq_len), dtype=torch.bool, device=x.device)
        is_pad = (curr_mask.squeeze(1).squeeze(-1) <= 0.5)          # (B, T_prime) bool
        padding_mask[:, 1:1 + T_prime] = is_pad

        encoded = self.transformer(seq, src_key_padding_mask=padding_mask)  # (B, T'+1, D)
        cls_out = encoded[:, 0]                                     # Lấy đặc trưng tại [CLS] token

        return self.head(cls_out)

In [ ]:
# Cell 7: Các hàm tiện ích hệ thống, tính toán chỉ số và Pipeline huấn luyện (train.py)
# =====================================================================
# 1. TIỆN ÍCH HỆ THỐNG: SEED, LOGGING & CHECKPOINT
# =====================================================================
def seed_everything(seed: int = 42) -> None:
    """Cố định seed để đảm bảo khả năng tái lập kết quả thực nghiệm."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def configure_logging(log_file: Path | str) -> None:
    """Cấu hình logging đồng thời ra Console và file train.log."""
    if hasattr(sys.stdout, "reconfigure"):
        try:
            sys.stdout.reconfigure(encoding="utf-8", errors="replace")
        except Exception:
            pass

    log_file = Path(log_file)
    log_file.parent.mkdir(parents=True, exist_ok=True)

    logger.setLevel(logging.INFO)
    logger.propagate = False
    for handler in logger.handlers[:]:
        handler.close()
        logger.removeHandler(handler)

    formatter = logging.Formatter("[%(asctime)s][%(levelname)s] %(message)s", datefmt="%Y-%m-%d %H:%M:%S")

    file_handler = logging.FileHandler(log_file, encoding="utf-8")
    file_handler.setFormatter(formatter)
    logger.addHandler(file_handler)

    stream_handler = logging.StreamHandler(sys.stdout)
    stream_handler.setFormatter(formatter)
    logger.addHandler(stream_handler)


def atomic_save(state: Any, path: str | Path) -> None:
    """Lưu tệp an toàn (Atomic Save qua file tạm) để tránh hỏng checkpoint khi bị ngắt đột ngột."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, tmp_file = tempfile.mkstemp(dir=path.parent, prefix=f".{path.name}.", suffix=".tmp")
    try:
        with os.fdopen(fd, "wb") as f:
            torch.save(state, f)
            f.flush()
            os.fsync(f.fileno())
        os.replace(tmp_file, path)
    finally:
        Path(tmp_file).unlink(missing_ok=True)


# =====================================================================
# 2. KHỞI TẠO MÔ HÌNH VÀ BỘ ĐIỀU TỐC LEARNING RATE
# =====================================================================
def build_model(cfg: TrainConfig) -> nn.Module:
    """Khởi tạo mô hình dựa trên TrainConfig."""
    model_name = cfg.model_name.lower().strip()
    if model_name in ("stgcn", "driverstgcn", "light"):
        logger.info("Khoi tao kien truc DriverSTGCN (Spatio-Temporal Graph CNN)...")
        model = DriverSTGCN(**cfg.get_stgcn_kwargs())
    elif model_name in ("transformer", "driverstgcntransformer", "trans"):
        logger.info("Khoi tao kien truc DriverSTGCNTransformer (STGCN + TransformerEncoder)...")
        model = DriverSTGCNTransformer(**cfg.get_transformer_kwargs())
    else:
        raise ValueError(f"Khong ho tro model_name='{cfg.model_name}'. Chon 'stgcn' hoac 'transformer'.")
    return model


def get_cosine_warmup_scheduler(
        optimizer: torch.optim.Optimizer,
        total_epochs: int,
        warmup_epochs: int = 5,
        min_lr_ratio: float = 0.05,
) -> LambdaLR:
    """Tạo bộ điều tốc Cosine Annealing kèm giai đoạn Linear Warmup."""
    warmup_epochs = max(0, min(warmup_epochs, total_epochs - 1))

    def lr_lambda(epoch: int) -> float:
        if epoch < warmup_epochs:
            return float(epoch + 1) / float(max(1, warmup_epochs))
        progress = float(epoch - warmup_epochs) / float(max(1, total_epochs - warmup_epochs - 1))
        progress = min(1.0, max(0.0, progress))
        return min_lr_ratio + (1.0 - min_lr_ratio) * 0.5 * (1.0 + math.cos(math.pi * progress))

    return LambdaLR(optimizer, lr_lambda=lr_lambda)


# =====================================================================
# 3. TÍNH TOÁN CHỈ SỐ: LOSS, ACCURACY, RECALL, F1, F2
# =====================================================================
def compute_metrics(
        y_true: np.ndarray,
        y_pred: np.ndarray,
        avg_loss: float,
        num_classes: int = 2,
) -> Dict[str, float]:
    """
    Tính toán chi tiết các chỉ số phân loại: Loss, Accuracy, Recall, F1-score, F2-score, Precision.
    - Với bài toán 2 lớp (0: Alert, 1: Drowsy): tập trung vào nhãn dương (1: Drowsy).
    - Với bài toán đa lớp (> 2 lớp): tính trung bình Macro trên tất cả các lớp.
    """
    total = len(y_true)
    if total == 0:
        return {
            "loss": float(avg_loss),
            "accuracy": 0.0,
            "precision": 0.0,
            "recall": 0.0,
            "f1": 0.0,
            "f2": 0.0,
            "n_samples": 0,
        }

    accuracy = float((y_true == y_pred).sum() / total)

    if num_classes == 2:
        tp = int(((y_pred == 1) & (y_true == 1)).sum())
        fp = int(((y_pred == 1) & (y_true == 0)).sum())
        fn = int(((y_pred == 0) & (y_true == 1)).sum())
        tn = int(((y_pred == 0) & (y_true == 0)).sum())

        precision = float(tp / max(1, tp + fp))
        recall = float(tp / max(1, tp + fn))
        f1 = float((2 * tp) / max(1, 2 * tp + fp + fn))
        # F2-score: beta=2 -> uu tien Recall cao hon Precision (giam bo sot tai xe ngu gat)
        f2 = float((5 * tp) / max(1, 5 * tp + 4 * fn + fp))
    else:
        # Macro average cho đa lớp
        precisions, recalls, f1s, f2s = [], [], [], []
        for c in range(num_classes):
            tp_c = int(((y_pred == c) & (y_true == c)).sum())
            fp_c = int(((y_pred == c) & (y_true != c)).sum())
            fn_c = int(((y_pred != c) & (y_true == c)).sum())

            p_c = tp_c / max(1, tp_c + fp_c)
            r_c = tp_c / max(1, tp_c + fn_c)
            f1_c = (2 * p_c * r_c) / max(1e-12, p_c + r_c)
            f2_c = (5 * p_c * r_c) / max(1e-12, 4 * p_c + r_c)

            precisions.append(p_c)
            recalls.append(r_c)
            f1s.append(f1_c)
            f2s.append(f2_c)

        precision = float(np.mean(precisions))
        recall = float(np.mean(recalls))
        f1 = float(np.mean(f1s))
        f2 = float(np.mean(f2s))

    return {
        "loss": float(avg_loss),
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "f2": f2,
        "n_samples": total,
    }


# =====================================================================
# 4. THỰC THI 1 EPOCH (TRAIN HOẶC EVALUATION)
# =====================================================================
def run_epoch(
        model: nn.Module,
        loader: DataLoader,
        criterion: nn.Module,
        device: torch.device,
        optimizer: Optional[torch.optim.Optimizer] = None,
        scaler: Optional[torch.amp.GradScaler] = None,
        amp: bool = True,
        max_grad_norm: float = 1.0,
        desc: str = "Train",
        num_classes: int = 2,
) -> Dict[str, float]:
    """Chạy 1 epoch đầy đủ qua DataLoader và trả về bộ chỉ số tổng hợp."""
    training = optimizer is not None
    model.train(training)

    total_loss = 0.0
    total_samples = 0
    all_targets = []
    all_preds = []

    progress = tqdm(loader, desc=desc, unit="batch", dynamic_ncols=True, leave=False)

    for batch in progress:
        # Nhận tuple 4 phần tử từ collate_gcn: (padded_landmarks, labels, lengths, metas)
        landmarks, labels, lengths = batch[0], batch[1], batch[2]
        landmarks = landmarks.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        lengths = lengths.to(device, non_blocking=True)
        count = labels.size(0)

        with torch.set_grad_enabled(training):
            with torch.autocast(device_type=device.type, enabled=amp and device.type == "cuda"):
                logits = model(landmarks, lengths=lengths)
                loss = criterion(logits, labels)

            if not torch.isfinite(loss):
                raise FloatingPointError(f"Gặp giá trị Loss không hợp lệ (NaN/Inf) tại {desc}")

            if training:
                optimizer.zero_grad(set_to_none=True)
                if scaler is not None and scaler.is_enabled():
                    scaler.scale(loss).backward()
                    scaler.unscale_(optimizer)
                    if max_grad_norm > 0:
                        nn.utils.clip_grad_norm_(model.parameters(), max_norm=max_grad_norm)
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    loss.backward()
                    if max_grad_norm > 0:
                        nn.utils.clip_grad_norm_(model.parameters(), max_norm=max_grad_norm)
                    optimizer.step()

        preds = logits.detach().argmax(dim=1)
        total_loss += loss.detach().item() * count
        total_samples += count

        all_targets.append(labels.detach().cpu().numpy())
        all_preds.append(preds.cpu().numpy())

        current_loss = total_loss / max(1, total_samples)
        progress.set_postfix(loss=f"{current_loss:.4f}")

    if total_samples == 0:
        return {
            "loss": 0.0,
            "accuracy": 0.0,
            "precision": 0.0,
            "recall": 0.0,
            "f1": 0.0,
            "f2": 0.0,
            "n_samples": 0,
        }

    y_true = np.concatenate(all_targets, axis=0)
    y_pred = np.concatenate(all_preds, axis=0)
    avg_loss = total_loss / total_samples

    return compute_metrics(y_true, y_pred, avg_loss, num_classes=num_classes)


# =====================================================================
# 5. GHI LOG TENSORBOARD ĐỒNG BỘ
# =====================================================================
def log_epoch_to_tensorboard(
        writer: SummaryWriter,
        train_metrics: Dict[str, float],
        val_metrics: Dict[str, float],
        current_lr: float,
        epoch: int,
) -> None:
    """Ghi nhận chi tiết các scalar Loss, Accuracy, Recall, F1, F2 cho cả Train & Val."""
    # 1. Các nhánh độc lập
    for metric_name in ("loss", "accuracy", "recall", "f1", "f2", "precision"):
        if metric_name in train_metrics:
            writer.add_scalar(f"train/{metric_name}", train_metrics[metric_name], epoch)
        if metric_name in val_metrics:
            writer.add_scalar(f"val/{metric_name}", val_metrics[metric_name], epoch)

    writer.add_scalar("train/learning_rate", current_lr, epoch)

    # 2. Đồ thị so sánh đối chiếu trực tiếp giữa Train và Validation
    for metric_name in ("loss", "accuracy", "recall", "f1", "f2"):
        writer.add_scalars(
            f"compare/{metric_name}",
            {"train": train_metrics.get(metric_name, 0.0), "val": val_metrics.get(metric_name, 0.0)},
            epoch,
        )
    writer.flush()


# =====================================================================
# 6. HÀM CHÍNH: QUẢN LÝ HUẤN LUYỆN & LƯU CHECKPOINT THEO F1
# =====================================================================
def train(cfg: Optional[TrainConfig] = None) -> nn.Module:
    """
    Hàm thực thi huấn luyện toàn bộ pipeline.
    Nhận đối tượng TrainConfig trực tiếp từ mã nguồn hoặc khởi tạo mặc định.
    """
    if cfg is None:
        cfg = TrainConfig()
    cfg.__post_init__()

    save_dir = Path(cfg.save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)
    configure_logging(save_dir / "train.log")

    log_dir = Path(cfg.log_dir) if cfg.log_dir else (save_dir / "tensorboard")
    log_dir.mkdir(parents=True, exist_ok=True)

    seed_everything(cfg.seed)
    device = torch.device(cfg.device if torch.cuda.is_available() or cfg.device == "cpu" else "cpu")

    logger.info("=" * 70)
    logger.info("BẮT ĐẦU HUẤN LUYỆN MÔ HÌNH DRIVER GUARDIAN GCN")
    logger.info("=" * 70)
    logger.info(f"Thiết bị sử dụng (Device) : {device}")
    logger.info(f"Kiến trúc mô hình (Model) : {cfg.model_name}")
    logger.info(f"Số lớp phân loại (Classes): {cfg.num_classes} ({cfg.class_names})")
    logger.info(f"Thư mục lưu Checkpoint     : {save_dir}")
    logger.info(f"Thư mục log TensorBoard   : {log_dir}")

    # Lưu lại file cấu hình json tại thư mục checkpoint
    cfg.save(save_dir / "config.json")

    # 1. Khởi tạo DataLoaders
    train_loader, val_loader, test_loader = build_gcn_dataloaders(**cfg.get_dataloader_kwargs())
    logger.info(
        f"Số lượng mẫu Dataset      : Train={len(train_loader.dataset)} | Val={len(val_loader.dataset)} | Test={len(test_loader.dataset)}")

    # 2. Khởi tạo Mô hình & Hàm Mất mát CrossEntropyLoss
    model = build_model(cfg).to(device)
    num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    logger.info(f"Tổng số tham số học được  : {num_params:,} parameters")

    criterion = nn.CrossEntropyLoss()
    optimizer = AdamW(
        (p for p in model.parameters() if p.requires_grad),
        lr=cfg.lr,
        weight_decay=cfg.weight_decay,
    )
    scheduler = get_cosine_warmup_scheduler(
        optimizer,
        total_epochs=cfg.epochs,
        warmup_epochs=cfg.warmup_epochs,
        min_lr_ratio=cfg.min_lr_ratio,
    )
    scaler = torch.amp.GradScaler("cuda", enabled=cfg.amp and device.type == "cuda")

    # 3. Quản lý khôi phục (Resume) nếu có
    start_epoch = 1
    best_epoch = 0
    best_metrics: Optional[Dict[str, float]] = None

    if cfg.resume:
        resume_path = Path(cfg.resume)
        if resume_path.is_file():
            logger.info(f"Nạp lại checkpoint từ: {resume_path}")
            checkpoint = torch.load(resume_path, map_location=device)
            model.load_state_dict(checkpoint["model_state_dict"])
            if "optimizer_state_dict" in checkpoint:
                optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
            if "scheduler_state_dict" in checkpoint:
                scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
            if "scaler_state_dict" in checkpoint and scaler is not None and checkpoint["scaler_state_dict"] is not None:
                scaler.load_state_dict(checkpoint["scaler_state_dict"])
            start_epoch = checkpoint.get("epoch", 0) + 1
            best_epoch = checkpoint.get("best_epoch", 0)
            best_metrics = checkpoint.get("best_metrics", None)
            logger.info(f"Đã khôi phục thành công! Tiếp tục từ Epoch {start_epoch} (Best Epoch trước đó: {best_epoch})")

    # 4. Vòng lặp huấn luyện từng Epoch & Đánh giá Checkpoint theo F1
    writer = SummaryWriter(str(log_dir))
    writer.add_text("Config", f"```json\n{json.dumps(cfg.to_dict(), indent=2, ensure_ascii=False)}\n```", 0)

    start_train_time = time.time()

    try:
        for epoch in range(start_epoch, cfg.epochs + 1):
            epoch_start = time.time()
            current_lr = float(optimizer.param_groups[0]["lr"])

            # Huấn luyện trên tập Train
            train_metrics = run_epoch(
                model=model,
                loader=train_loader,
                criterion=criterion,
                device=device,
                optimizer=optimizer,
                scaler=scaler,
                amp=cfg.amp,
                max_grad_norm=cfg.max_grad_norm,
                desc=f"Epoch {epoch:02d}/{cfg.epochs:02d} [Train]",
                num_classes=cfg.num_classes,
            )

            # Đánh giá trên tập Validation
            val_metrics = run_epoch(
                model=model,
                loader=val_loader,
                criterion=criterion,
                device=device,
                optimizer=None,
                scaler=None,
                amp=cfg.amp,
                max_grad_norm=cfg.max_grad_norm,
                desc=f"Epoch {epoch:02d}/{cfg.epochs:02d} [Val]",
                num_classes=cfg.num_classes,
            )

            scheduler.step()
            epoch_duration = time.time() - epoch_start

            # Ghi TensorBoard
            log_epoch_to_tensorboard(writer, train_metrics, val_metrics, current_lr, epoch)

            # Đánh giá tiêu chí lưu Checkpoint tốt nhất: DỰA TRÊN CHỈ SỐ F1 CỦA TẬP VALIDATION
            is_best = False
            if best_metrics is None:
                is_best = True
            else:
                # Ưu tiên F1 cao hơn; nếu F1 bằng nhau xét tiếp F2 và sau đó là Loss thấp hơn
                val_score = (val_metrics["f1"], val_metrics["f2"], -val_metrics["loss"])
                best_score = (best_metrics["f1"], best_metrics["f2"], -best_metrics["loss"])
                if val_score > best_score:
                    is_best = True

            if is_best:
                best_epoch = epoch
                best_metrics = val_metrics
                best_state = {
                    "format_version": 1,
                    "epoch": best_epoch,
                    "model_name": cfg.model_name,
                    "model_state_dict": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                    "val_metrics": best_metrics,
                    "config": cfg.to_dict(),
                }
                atomic_save(best_state, save_dir / "best.pt")
                with open(save_dir / "best_metrics.json", "w", encoding="utf-8") as f:
                    json.dump(best_metrics, f, indent=4, ensure_ascii=False)

            # Lưu checkpoint cuối cùng sau mỗi epoch để hỗ trợ resume
            last_state = {
                "format_version": 1,
                "epoch": epoch,
                "model_name": cfg.model_name,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "scaler_state_dict": scaler.state_dict() if scaler else None,
                "best_epoch": best_epoch,
                "best_metrics": best_metrics,
                "config": cfg.to_dict(),
            }
            atomic_save(last_state, save_dir / "last.pt")

            # In thông tin tổng kết Epoch
            best_tag = " >>> [BEST CHECKPOINT SAVED] <<<" if is_best else ""
            logger.info(
                f"Epoch {epoch:02d}/{cfg.epochs:02d} ({epoch_duration:.1f}s) | LR: {current_lr:.6g}\n"
                f"  - Train: Loss={train_metrics['loss']:.4f} | Acc={train_metrics['accuracy']:.4f} | Recall={train_metrics['recall']:.4f} | F1={train_metrics['f1']:.4f} | F2={train_metrics['f2']:.4f}\n"
                f"  - Val  : Loss={val_metrics['loss']:.4f} | Acc={val_metrics['accuracy']:.4f} | Recall={val_metrics['recall']:.4f} | F1={val_metrics['f1']:.4f} | F2={val_metrics['f2']:.4f}{best_tag}"
            )

    except KeyboardInterrupt:
        logger.warning("\nQuá trình huấn luyện bị ngắt bởi người dùng. Checkpoint 'last.pt' đã được bảo lưu an toàn.")
    finally:
        total_time = time.time() - start_train_time
        logger.info(f"\nHoàn tất huấn luyện sau {total_time / 60:.2f} phút.")
        if best_metrics:
            logger.info(f"Checkpoint tốt nhất tại Epoch {best_epoch}:")
            logger.info(f"  - Val F1      : {best_metrics['f1']:.4f}")
            logger.info(f"  - Val F2      : {best_metrics['f2']:.4f}")
            logger.info(f"  - Val Recall  : {best_metrics['recall']:.4f}")
            logger.info(f"  - Val Accuracy: {best_metrics['accuracy']:.4f}")
            logger.info(f"  - Val Loss    : {best_metrics['loss']:.4f}")

        # 5. Đánh giá kiểm thử cuối cùng trên tập Test bằng model tốt nhất
        best_pt_path = save_dir / "best.pt"
        if best_pt_path.is_file() and len(test_loader.dataset) > 0:
            logger.info("\nĐang tải best.pt để đánh giá nghiệm thu trên tập Test...")
            checkpoint = torch.load(best_pt_path, map_location=device)
            model.load_state_dict(checkpoint["model_state_dict"])
            test_metrics = run_epoch(
                model=model,
                loader=test_loader,
                criterion=criterion,
                device=device,
                optimizer=None,
                scaler=None,
                amp=cfg.amp,
                desc="Final Test",
                num_classes=cfg.num_classes,
            )
            for m_name, m_val in test_metrics.items():
                writer.add_scalar(f"test/{m_name}", m_val, best_epoch)
            writer.flush()
            with open(save_dir / "test_metrics.json", "w", encoding="utf-8") as f:
                json.dump(test_metrics, f, indent=4, ensure_ascii=False)
            logger.info(
                f"Kết quả Test (Best Epoch {best_epoch}): "
                f"Loss={test_metrics['loss']:.4f} | Acc={test_metrics['accuracy']:.4f} | "
                f"Recall={test_metrics['recall']:.4f} | F1={test_metrics['f1']:.4f} | F2={test_metrics['f2']:.4f}"
            )

        writer.close()

    return model

In [ ]:
# Cell 8: Khởi tạo cấu hình TrainConfig và bắt đầu huấn luyện
config = TrainConfig()
trained_model = train(config)